In [0]:
print(spark.version)

In [0]:
%pip install graphframes-py==0.12.2

In [0]:
print("Spark version:", spark.version)

try:
    from graphframes import GraphFrame
    print("GraphFrames: AVAILABLE")
except Exception as e:
    print("GraphFrames: NOT AVAILABLE")
    print("Error:", str(e))

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from graphframes import GraphFrame


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
SILVER_ACCOUNTS_TABLE = f"{CATALOG}.{SCHEMA}.silver_accounts"

GRAPH_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.graph_results"

GRAPH_ACCOUNT_FEATURES_TABLE = (
    f"{CATALOG}.{SCHEMA}.graph_account_features"
)

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

GRAPH_RESULTS_PATH = (
    f"{S3_DELTA_PATH}/graph_results"
)

GRAPH_ACCOUNT_FEATURES_PATH = (
    f"{S3_DELTA_PATH}/graph_account_features"
)


# ============================================================
# 2. GRAPH RULE CONFIGURATION
# ============================================================

CYCLE_TIME_WINDOW = 2
CYCLE_RULE_SCORE = 50


print("Graph configuration loaded.")
print(f"Silver Transactions : {SILVER_TX_TABLE}")
print(f"Silver Accounts     : {SILVER_ACCOUNTS_TABLE}")
print(f"Graph Results       : {GRAPH_RESULTS_TABLE}")
print(f"Graph Features      : {GRAPH_ACCOUNT_FEATURES_TABLE}")


# COMMAND ----------

# ============================================================
# 3. READ CURRENT SILVER DATA
# ============================================================
#
# IMPORTANT:
# Silver is already incremental.
#
# Here we read the COMPLETE current Silver table because
# graph calculations must consider:
#
#   OLD TRANSACTIONS + NEW TRANSACTIONS
#
# A new transaction can complete a cycle involving older
# transactions.
# ============================================================

transactions_df = spark.table(SILVER_TX_TABLE)

accounts_df = spark.table(SILVER_ACCOUNTS_TABLE)

print(
    f"Silver transactions: {transactions_df.count()}"
)

print(
    f"Silver accounts: {accounts_df.count()}"
)


# COMMAND ----------

# ============================================================
# 4. CREATE GRAPH VERTICES
# ============================================================

vertices_df = (
    accounts_df
    .select(
        F.col("account_id")
        .cast("string")
        .alias("id")
    )
    .filter(
        F.col("id").isNotNull()
    )
    .dropDuplicates(["id"])
)

print(
    f"Graph vertices: {vertices_df.count()}"
)


# COMMAND ----------

# ============================================================
# 5. CREATE GRAPH EDGES
# ============================================================

edges_df = (
    transactions_df
    .select(
        F.col("sender_account_id")
        .cast("string")
        .alias("src"),

        F.col("receiver_account_id")
        .cast("string")
        .alias("dst"),

        F.col("tx_id")
        .cast("long")
        .alias("tx_id"),

        F.col("tx_amount")
        .cast("double")
        .alias("tx_amount"),

        F.col("tx_type")
        .alias("tx_type"),

        F.col("event_time")
        .cast("long")
        .alias("event_time")
    )

    .filter(
        F.col("src").isNotNull()
    )

    .filter(
        F.col("dst").isNotNull()
    )

    .filter(
        F.col("tx_id").isNotNull()
    )

    .dropDuplicates(
        [
            "src",
            "dst",
            "tx_id",
            "event_time"
        ]
    )
)

print(
    f"Graph edges: {edges_df.count()}"
)


# COMMAND ----------

# ============================================================
# 6. CREATE GRAPHFRAME
# ============================================================

g = GraphFrame(
    vertices_df,
    edges_df
)

print("AML GraphFrame created successfully.")


# COMMAND ----------

# ============================================================
# 7. ACCOUNT GRAPH FEATURES
# ============================================================

in_degree_df = (
    g.inDegrees
    .withColumnRenamed(
        "id",
        "account_id"
    )
    .withColumnRenamed(
        "inDegree",
        "in_degree"
    )
)

out_degree_df = (
    g.outDegrees
    .withColumnRenamed(
        "id",
        "account_id"
    )
    .withColumnRenamed(
        "outDegree",
        "out_degree"
    )
)


# COMMAND ----------

# ============================================================
# 8. BUILD ACCOUNT GRAPH FEATURES
# ============================================================

graph_account_features_df = (
    vertices_df

    .withColumn(
        "account_id",
        F.col("id").cast("long")
    )

    .drop("id")

    .join(
        in_degree_df,
        on="account_id",
        how="left"
    )

    .join(
        out_degree_df,
        on="account_id",
        how="left"
    )

    .fillna(
        {
            "in_degree": 0,
            "out_degree": 0
        }
    )

    .withColumn(
        "total_degree",
        F.col("in_degree") +
        F.col("out_degree")
    )

    .withColumn(
        "graph_processed_timestamp",
        F.current_timestamp()
    )
)


# COMMAND ----------

# ============================================================
# 9. SAVE CURRENT GRAPH ACCOUNT FEATURES
# ============================================================
#
# OVERWRITE IS INTENTIONAL HERE.
#
# This table represents the CURRENT graph state.
#
# Silver transactions remain incremental.
# Graph features are recalculated from the current Silver state.
# ============================================================

(
    graph_account_features_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        GRAPH_ACCOUNT_FEATURES_PATH
    )
    .saveAsTable(
        GRAPH_ACCOUNT_FEATURES_TABLE
    )
)

print("Graph account features updated.")


# COMMAND ----------

# ============================================================
# 10. CYCLE DETECTION
# ============================================================

ab = edges_df.alias("ab")
bc = edges_df.alias("bc")
ca = edges_df.alias("ca")


# COMMAND ----------

# ============================================================
# 11. FIND A -> B -> C
# ============================================================

ab_bc = (
    ab.join(
        bc,
        F.col("ab.dst") == F.col("bc.src"),
        "inner"
    )

    .select(
        F.col("ab.src").alias("account_a"),
        F.col("ab.dst").alias("account_b"),
        F.col("bc.dst").alias("account_c"),

        F.col("ab.tx_id").alias("tx_ab"),
        F.col("bc.tx_id").alias("tx_bc"),

        F.col("ab.tx_amount").alias("amount_ab"),
        F.col("bc.tx_amount").alias("amount_bc"),

        F.col("ab.event_time").alias("time_ab"),
        F.col("bc.event_time").alias("time_bc")
    )
)


# COMMAND ----------

# ============================================================
# 12. CLOSE THE CYCLE C -> A
# ============================================================

ab_bc_ca = (
    ab_bc.alias("path")

    .join(
        ca,
        (
            (F.col("path.account_c") == F.col("ca.src")) &
            (F.col("ca.dst") == F.col("path.account_a"))
        ),
        "inner"
    )

    .select(
        F.col("path.account_a").alias("account_a"),
        F.col("path.account_b").alias("account_b"),
        F.col("path.account_c").alias("account_c"),

        F.col("path.tx_ab").alias("tx_ab"),
        F.col("path.tx_bc").alias("tx_bc"),
        F.col("ca.tx_id").alias("tx_ca"),

        F.col("path.amount_ab").alias("amount_ab"),
        F.col("path.amount_bc").alias("amount_bc"),
        F.col("ca.tx_amount").alias("amount_ca"),

        F.col("path.time_ab").alias("time_ab"),
        F.col("path.time_bc").alias("time_bc"),
        F.col("ca.event_time").alias("time_ca")
    )
)


# COMMAND ----------

# ============================================================
# 13. FILTER VALID CYCLES
# ============================================================

cycles = (
    ab_bc_ca

    .filter(
        (F.col("account_a") != F.col("account_b")) &
        (F.col("account_b") != F.col("account_c")) &
        (F.col("account_c") != F.col("account_a"))
    )

    .filter(
        (F.col("tx_ab") != F.col("tx_bc")) &
        (F.col("tx_bc") != F.col("tx_ca")) &
        (F.col("tx_ca") != F.col("tx_ab"))
    )

    .withColumn(
        "min_event_time",
        F.least(
            "time_ab",
            "time_bc",
            "time_ca"
        )
    )

    .withColumn(
        "max_event_time",
        F.greatest(
            "time_ab",
            "time_bc",
            "time_ca"
        )
    )

    .withColumn(
        "cycle_time_span",
        F.col("max_event_time") -
        F.col("min_event_time")
    )

    .filter(
        F.col("cycle_time_span") <= CYCLE_TIME_WINDOW
    )
)


# COMMAND ----------

# ============================================================
# 14. CREATE UNIQUE CYCLE ID
# ============================================================

cycles_unique = (
    cycles

    .withColumn(
        "cycle_tx_ids",
        F.array_sort(
            F.array(
                F.col("tx_ab"),
                F.col("tx_bc"),
                F.col("tx_ca")
            )
        )
    )

    .withColumn(
        "cycle_id",
        F.concat_ws(
            "-",
            F.col("cycle_tx_ids")
        )
    )

    .dropDuplicates(
        ["cycle_id"]
    )
)


# COMMAND ----------

# ============================================================
# 15. CREATE GRAPH RESULTS
# ============================================================

graph_results_df = (
    cycles_unique

    .select(
        "cycle_id",
        "account_a",
        "account_b",
        "account_c",
        "cycle_time_span",
        "graph_evidence"
        if "graph_evidence" in cycles_unique.columns
        else F.lit(None).cast("string").alias("graph_evidence"),

        F.explode(
            F.array(
                "tx_ab",
                "tx_bc",
                "tx_ca"
            )
        ).alias("tx_id")
    )

    .withColumn(
        "rule_id",
        F.lit("R005")
    )

    .withColumn(
        "rule_name",
        F.lit("CYCLE_DETECTION")
    )

    .withColumn(
        "rule_category",
        F.lit("GRAPH")
    )

    .withColumn(
        "rule_triggered",
        F.lit(True)
    )

    .withColumn(
        "graph_rule_score",
        F.lit(CYCLE_RULE_SCORE)
    )

    .withColumn(
        "rule_version",
        F.lit("v1.0")
    )

    .withColumn(
        "graph_execution_timestamp",
        F.current_timestamp()
    )
)


# COMMAND ----------

# ============================================================
# 16. ADD GRAPH EVIDENCE
# ============================================================

graph_results_df = (
    graph_results_df

    .withColumn(
        "graph_evidence",
        F.concat(
            F.lit("Circular transaction path detected: "),
            F.col("account_a"),
            F.lit(" -> "),
            F.col("account_b"),
            F.lit(" -> "),
            F.col("account_c"),
            F.lit(" -> "),
            F.col("account_a")
        )
    )
)


# COMMAND ----------

# ============================================================
# 17. SAVE CURRENT GRAPH RESULTS
# ============================================================
#
# OVERWRITE is intentional.
#
# graph_results represents the current graph analysis.
# Recalculate from the complete Silver transaction state.
# ============================================================

(
    graph_results_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        GRAPH_RESULTS_PATH
    )
    .saveAsTable(
        GRAPH_RESULTS_TABLE
    )
)

print("Graph results updated.")
print(f"Unity Catalog table : {GRAPH_RESULTS_TABLE}")
print(f"S3 location         : {GRAPH_RESULTS_PATH}")


# COMMAND ----------

# ============================================================
# 18. VALIDATION
# ============================================================

display(
    graph_results_df.limit(20)
)


# COMMAND ----------

# ============================================================
# 19. SUMMARY
# ============================================================

print("==============================================")
print("GRAPH PROCESSING COMPLETED")
print("==============================================")
print(f"Transactions : {transactions_df.count()}")
print(f"Accounts     : {accounts_df.count()}")
print(f"Graph edges  : {edges_df.count()}")
print(f"Graph results: {graph_results_df.count()}")
print("==============================================")